In [57]:
import numpy as np
import pandas as pd
import re
import string
import pickle

In [58]:
def remove_punctuation(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, "")
    return text

In [59]:
with open('../static/model/supported_vector_machine.pkl', 'rb') as file:
    model = pickle.load(file)

In [60]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [61]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
token = vocab[0].tolist()

In [62]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

## Preprocessing

In [63]:
def preprocess(text):
    data = pd.DataFrame([text], columns=["tweet"])

    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()) )

    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^http:?:\/\/.*[\r\n]*', '', x, flags=re.MULTILINE) for x in x.split()) )

    data["tweet"] = data["tweet"].apply(lambda x: " ".join(remove_punctuation(x) for x in x.split()) )

    data["tweet"] = data["tweet"].str.replace(r'\d+', '', regex=True)

    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw) )

    data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()) )
    return data["tweet"]


## Vectorizetion

In [64]:
def vectorizer(ds, vocabulary):
    vecorized_1st = []

    for sentence in ds: 
        sentence_1st = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_1st[i] = 1

        vecorized_1st.append(sentence_1st)

    vecorized_1st_new = np.array(vecorized_1st, dtype=np.float32)
    return vecorized_1st_new

In [65]:
def get_prediction(vectorized_text):
    prediction = model.predict(vectorized_text)
    if prediction == 1:
        return "Negative"
    else:
        return "Positive"

In [68]:
txt = "good service!"
preprocessed_txt = preprocess(txt)
vectorized_txt = vectorizer(preprocessed_txt, token)
prediction = get_prediction(vectorized_txt)
prediction

'Positive'